# Week 4 — Policy Uncertainty Analysis
Standalone notebook. Reads Week 3 disk outputs; never re-runs rolling GARCH/VAR.

**Markets processed in isolation:** China (benchmark) → US (comparison).  
Comparison occurs only in Stage C (cross-market interpretation).

**Required disk inputs per market** (`outputs/{market}/`):
- `rolling_tsi.csv` — index: `date`, column: `TSI` (+ `NET_*`, `TO_*`, `FROM_*`)
- `garch_volatility_panel.csv` — daily conditional σ, columns = GICS sector names

**New raw data** (`data/{market}/`):
- EPU file — Economic Policy Uncertainty index, monthly (filename/columns per `MARKET_CONFIGS`)
- TPU file — Trade Policy Uncertainty index, monthly (filename/columns per `MARKET_CONFIGS`)

File names and column schemas are fully specified in `MARKET_CONFIGS` — no hardcoded globals.

## 0.0 — Setup

Run this in terminal before importing:

```bash
pip install numpy ipykernel
pip install pandas ipykernel
pip install statsmodels ipykernel
pip install arch ipykernel
pip install openpyxl
pip install matplotlib 
pip install seaborn
```

In [1]:
import warnings, logging
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from statsmodels.tsa.api import VAR
from statsmodels.tsa.stattools import adfuller

warnings.filterwarnings('ignore')

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
)
log = logging.getLogger(__name__)

In [ ]:
BASE_DIR    = Path().resolve()
DATA_DIR    = BASE_DIR / 'data'
OUTPUTS_DIR = BASE_DIR / 'outputs'
LOGS_DIR    = BASE_DIR / 'logs'
ISSUE_LOG   = LOGS_DIR / 'issue_log.txt'
for d in [OUTPUTS_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Hyperparameters (inherited from Week 3 where shared) ──────────────────────
ADF_MAXLAG   = 12
ADF_PVAL_THR = 0.05
HORIZON      = 10       # GFEVD forecast horizon
IRF_REPS     = 1000     # bootstrap replications for IRF CIs
IRF_HORIZON  = 12       # months ahead for Route 1 IRF
FDR_ALPHA    = 0.05     # Benjamini-Hochberg FDR level

# ── Market registry — all file/column names live here, nowhere else ───────────
MARKETS = ['china', 'us']

MARKET_CONFIGS = {
    'china': {
        'epu': {
            'epu_file': 'China_EPU.csv',                        # file name of EPU data
            'epu_sheet': [0,1,2],                               # sheet numbers inside epu_file
            'epu_col':  'EPU',                                  # EPU column name
            'is_split_date': True,                              # EPU and TPU dates are split to year and month columns
            'epu_year_col': 'year',                             # year column inside epu_file
            'epu_month_col': 'month',                           # month column inside epu_file
        },
        'tpu': {
            'tpu_file': 'China_Mainland_Paper_TPU.csv',         # file name of TPU data
            'tpu_sheet': 3,                                     # sheet number inside tpu_file
            'tpu_col':  'TPU',                                  # TPU column name
            'is_split_date': True,                              # EPU and TPU dates are split to year and month columns
            'tpu_year_col': 'year',                             # year column inside tpu_file
            'tpu_month_col': 'month',                           # month column inside tpu_file
        }
    },
    'us': {
        'epu': {
            'epu_file': 'US_EPU.csv',                           # file name of EPU data
            'epu_sheet': 0,                                     # sheet number inside epu_file
            'epu_col':  'News_Based_Policy_Uncert_Index',       # EPU column name
            'is_split_date': True,                              # EPU and TPU dates are split to year and month columns'
            'epu_year_col': 'Year',                             # year column inside epu_file
            'epu_month_col': 'Month',                           # month column inside epu_file
        },
        'tpu': {
            'tpu_file': 'US_TPU.csv',                           # file name of TPU data
            'tpu_sheet': 2,                                     # sheet number inside tpu_file
            'tpu_col':  'TPU',                                  # numeric column inside tpu_file
            'is_split_date': False,                             # EPU and TPU dates are split to year and month columns
            'date_col': 'DATE',                                 # date column inside tpu_file
        }
    },
}

GICS_SECTORS = [
    'Energy', 'Materials', 'Industrials',
    'Consumer Discretionary', 'Consumer Staples', 'Health Care',
    'Financials', 'Information Technology', 'Communication Services',
    'Utilities', 'Real Estate',
]

log.info('Configuration loaded. Markets: %s', MARKETS)

2026-06-26 14:21:23  INFO      Configuration loaded. Markets: ['china', 'us']


## Shared utilities

In [3]:
def _log_issue(market, context, message):
    ts    = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    entry = f'[{ts}] [{market.upper()}] [{context}] {message}\n'
    with open(ISSUE_LOG, 'a') as fh:
        fh.write(entry)
    log.warning(entry.strip())


def adf_test(series, maxlag=ADF_MAXLAG, pval_thr=ADF_PVAL_THR):
    """Return (stationary: bool, p_value: float)."""
    s = series.dropna()
    if len(s) < 20:
        return False, np.nan
    stat, pval, *_ = adfuller(s, maxlag=maxlag, autolag='AIC', regression='c')
    return pval < pval_thr, round(pval, 6)


def log_diff(series):
    """Log-first-difference a monthly Series; drops the leading NaN."""
    if (series <= 0).any():
        raise ValueError(f'Non-positive values in series {series.name!r}; cannot log-transform.')
    return np.log(series).diff(1).dropna()


def make_stationary(series, market, label):
    """ADF-gate: return level if stationary, log-diff otherwise. Logs decision."""
    stat, pval = adf_test(series)
    if stat:
        log.info('[%s] %s stationary at level (ADF p=%.4f) → using level', market, label, pval)
        return series, label, pval
    log.info('[%s] %s non-stationary (ADF p=%.4f) → log-differencing', market, label, pval)
    diff = log_diff(series)
    stat2, pval2 = adf_test(diff)
    if not stat2:
        _log_issue(market, 'ADF', f'{label} still non-stationary after log-diff (p={pval2:.4f})')
    return diff, f'Δln_{label}', pval2


# ── GFEVD (Pesaran-Shin 1998, order-invariant) ────────────────────────────────
# Exact implementation carried over from Week 3 (compute_gfevd / row_norm /
# directional_measures) to guarantee consistent results in Route 2.

def compute_gfevd(var_res, H):
    A   = np.asarray(var_res.irf(periods=H).irfs)   # (H+1, n, n)
    Sig = np.asarray(var_res.sigma_u)
    n   = Sig.shape[0]
    sig_diag = np.diag(Sig)
    theta = np.zeros((n, n))
    for i in range(n):
        denom = sum(A[h, i, :] @ Sig @ A[h, i, :].T for h in range(H + 1))
        for j in range(n):
            numer = (1. / sig_diag[j]) * sum(
                (A[h, i, :] @ Sig[:, j]) ** 2 for h in range(H + 1)
            )
            theta[i, j] = numer / denom if denom > 1e-10 else 0.
    return theta


def row_norm(theta):
    rs = theta.sum(axis=1, keepdims=True)
    return theta / np.where(rs == 0, 1., rs)


def directional_measures(th_star, names):
    off = th_star.copy()
    np.fill_diagonal(off, 0.)
    TO   = off.sum(axis=0)
    FROM = off.sum(axis=1)
    NET  = TO - FROM
    OWN  = np.diag(th_star)
    tsi  = off.sum() / len(names) * 100
    df   = pd.DataFrame(
        {'TO': TO * 100, 'FROM': FROM * 100, 'NET': NET * 100, 'OWN': OWN * 100},
        index=names,
    ).round(4)
    return df, tsi


# ── Benjamini-Hochberg FDR correction ─────────────────────────────────────────
def bh_correct(p_values, alpha=FDR_ALPHA):
    """Return array of booleans: True where H0 is rejected after BH correction."""
    p = np.asarray(p_values, dtype=float)
    m = len(p)
    if m == 0:
        return np.array([], dtype=bool)
    ranks  = np.argsort(p) + 1          # rank 1 = smallest p
    order  = np.argsort(p)
    thresholds = (ranks / m) * alpha
    # Find largest rank k where p_(k) <= (k/m)*alpha
    below  = p[order] <= thresholds
    if not below.any():
        return np.zeros(m, dtype=bool)
    k_max  = np.where(below)[0].max()
    reject = np.zeros(m, dtype=bool)
    reject[order[:k_max + 1]] = True
    return reject

## Stage A — Data preparation pipeline
`prepare_market_data(market_id, cfg)` is the black-box function for Steps A0–A3.  
It reads Week 3 outputs and new EPU/TPU files using `MARKET_CONFIGS[market_id]`; returns two aligned monthly panels.

In [4]:
def prepare_market_data(market_id, cfg):
    """
    Steps A0–A3 for one market.

    Parameters
    ----------
    market_id : str
        One of MARKETS ('china', 'us').
    cfg : dict
        MARKET_CONFIGS[market_id] — file names and column labels.

    Returns
    -------
    route1_panel : pd.DataFrame
        Monthly, columns = [tci_col, 'ΔEPU', 'ΔTPU']
    route2_panel : pd.DataFrame
        Monthly, columns = 11 sector vols + ['ΔEPU', 'ΔTPU']
    meta : dict
        Diagnostic metadata (windows, ADF results, column labels)
    """
    mid   = market_id.lower()
    out_m = OUTPUTS_DIR / mid
    dat_m = DATA_DIR    / mid
    log.info('[%s] ── Stage A: loading data ──', mid.upper())

    # ── A0: Load inputs ────────────────────────────────────────────────────────
    # rolling_tsi.csv: index='date', column='TSI'
    tsi_path = out_m / 'rolling_tsi.csv'
    if not tsi_path.exists():
        raise FileNotFoundError(f'Missing: {tsi_path}')
    daily_tsi = pd.read_csv(tsi_path, index_col='date', parse_dates=True)
    if daily_tsi.index.duplicated().any():
        n_dup = daily_tsi.index.duplicated().sum()
        _log_issue(mid, 'A0', f'rolling_tsi.csv: {n_dup} duplicate dates — keeping last')
        daily_tsi = daily_tsi[~daily_tsi.index.duplicated(keep='last')]

    # garch_volatility_panel.csv: index=date, columns=GICS sector names
    vol_path = out_m / 'garch_volatility_panel.csv'
    if not vol_path.exists():
        raise FileNotFoundError(f'Missing: {vol_path}')
    daily_vol = pd.read_csv(vol_path, index_col=0, parse_dates=True)
    # Keep only the 11 canonical sector columns
    avail_secs   = [s for s in GICS_SECTORS if s in daily_vol.columns]
    missing_secs = [s for s in GICS_SECTORS if s not in daily_vol.columns]
    if missing_secs:
        _log_issue(mid, 'A0', f'Sectors missing from vol panel: {missing_secs}')
    daily_vol = daily_vol[avail_secs].copy()

    # EPU / TPU — read via per-market config (filename + column names)
    epu_path = dat_m / cfg['epu_file']
    tpu_path = dat_m / cfg['tpu_file']
    for p in [epu_path, tpu_path]:
        if not p.exists():
            raise FileNotFoundError(f'Missing policy file: {p}')

    epu_raw = pd.read_csv(epu_path)
    tpu_raw = pd.read_csv(tpu_path)

    epu_date_col = cfg['date_col']
    tpu_date_col = cfg['date_col']
    epu_val_col  = cfg['epu_col']
    tpu_val_col  = cfg['tpu_col']

    # Validate required columns exist
    for df, dcol, vcol, label in [
        (epu_raw, epu_date_col, epu_val_col, 'EPU'),
        (tpu_raw, tpu_date_col, tpu_val_col, 'TPU'),
    ]:
        for col in [dcol, vcol]:
            if col not in df.columns:
                raise KeyError(
                    f'[{mid.upper()}] {label} file missing column "{col}". '
                    f'Available: {list(df.columns)}'
                )
        try:
            pd.to_datetime(df[dcol])
        except Exception as e:
            raise ValueError(f'{label} date column "{dcol}" not parseable: {e}')

    log.info('[%s] A0 loaded: TSI=%d rows, vol=%s, EPU=%d rows, TPU=%d rows',
             mid.upper(), len(daily_tsi), daily_vol.shape, len(epu_raw), len(tpu_raw))

    # ── A1: Frequency alignment — daily → monthly (within-month mean) ──────────
    # TCI
    daily_tsi.index = pd.to_datetime(daily_tsi.index)
    daily_tsi['_month'] = daily_tsi.index.to_period('M').to_timestamp()
    tci_monthly = (
        daily_tsi.groupby('_month')['TSI']
        .mean()
        .rename('TCI')
    )
    tci_monthly.index.name = 'date'

    # Sector volatilities
    daily_vol.index = pd.to_datetime(daily_vol.index)
    daily_vol['_month'] = daily_vol.index.to_period('M').to_timestamp()
    vol_monthly = daily_vol.groupby('_month')[avail_secs].mean()
    vol_monthly.index.name = 'date'

    # Policy — parse to month-start timestamps using config column names
    epu_raw[epu_date_col] = pd.to_datetime(epu_raw[epu_date_col]).dt.to_period('M').dt.to_timestamp()
    tpu_raw[tpu_date_col] = pd.to_datetime(tpu_raw[tpu_date_col]).dt.to_period('M').dt.to_timestamp()
    epu_monthly = epu_raw.set_index(epu_date_col)[[epu_val_col]].rename(columns={epu_val_col: 'EPU'})
    tpu_monthly = tpu_raw.set_index(tpu_date_col)[[tpu_val_col]].rename(columns={tpu_val_col: 'TPU'})
    epu_monthly.index.name = 'date'
    tpu_monthly.index.name = 'date'

    log.info('[%s] A1 aligned: TCI=%d mo, vol=%d mo, EPU=%d mo, TPU=%d mo',
             mid.upper(), len(tci_monthly), len(vol_monthly), len(epu_monthly), len(tpu_monthly))

    # ── A2: Stationarity transforms ────────────────────────────────────────────
    # Policy: unconditional log-difference (level series always integrated)
    if (epu_monthly['EPU'] <= 0).any():
        raise ValueError(f'[{mid}] EPU contains non-positive values.')
    if (tpu_monthly['TPU'] <= 0).any():
        raise ValueError(f'[{mid}] TPU contains non-positive values.')

    delta_epu = log_diff(epu_monthly['EPU']).rename('ΔEPU')
    delta_tpu = log_diff(tpu_monthly['TPU']).rename('ΔTPU')

    # Validate post-transform stationarity
    adf_results = {}
    for s, label in [(delta_epu, 'ΔEPU'), (delta_tpu, 'ΔTPU')]:
        stat, pval = adf_test(s)
        adf_results[label] = {'stationary': stat, 'p': pval}
        if not stat:
            _log_issue(mid, 'ADF', f'{label} not stationary after log-diff (p={pval:.4f})')
        log.info('[%s] ADF %s: p=%.4f stationary=%s', mid.upper(), label, pval, stat)

    # TCI: ADF-gated decision
    tci_final, tci_col, tci_pval = make_stationary(tci_monthly, mid, 'TCI')
    adf_results['TCI'] = {'stationary': tci_pval < ADF_PVAL_THR, 'p': tci_pval}

    # Sector vols: ADF-gated per sector
    vol_final_cols = {}
    for sec in avail_secs:
        s_final, s_col, s_pval = make_stationary(vol_monthly[sec], mid, sec)
        vol_final_cols[s_col] = s_final
        adf_results[sec] = {'stationary': s_pval < ADF_PVAL_THR, 'p': s_pval}
    vol_final = pd.DataFrame(vol_final_cols)

    # ── A3: Merge alignment — inner join on common monthly window ───────────────
    route1_panel = (
        pd.concat([tci_final.rename(tci_col), delta_epu, delta_tpu], axis=1)
        .dropna()
    )
    route2_panel = (
        pd.concat([vol_final, delta_epu, delta_tpu], axis=1)
        .dropna()
    )

    assert route1_panel.shape[1] == 3, 'Route 1 panel must have exactly 3 columns'
    assert route2_panel.shape[1] == len(avail_secs) + 2, 'Route 2 panel column count mismatch'
    assert not route1_panel.index.duplicated().any(), 'Route 1 panel has duplicate dates'
    assert not route2_panel.index.duplicated().any(), 'Route 2 panel has duplicate dates'

    log.info('[%s] A3 Route1 window: %s → %s  N=%d months  cols=%s',
             mid.upper(),
             route1_panel.index[0].date(), route1_panel.index[-1].date(),
             len(route1_panel), list(route1_panel.columns))
    log.info('[%s] A3 Route2 window: %s → %s  N=%d months  nodes=%d',
             mid.upper(),
             route2_panel.index[0].date(), route2_panel.index[-1].date(),
             len(route2_panel), route2_panel.shape[1])

    meta = {
        'tci_col':    tci_col,
        'adf':        adf_results,
        'r1_window':  (route1_panel.index[0], route1_panel.index[-1]),
        'r2_window':  (route2_panel.index[0], route2_panel.index[-1]),
        'r1_N':       len(route1_panel),
        'r2_N':       len(route2_panel),
        'avail_secs': avail_secs,
    }
    return route1_panel, route2_panel, meta

## Stage B — Route 1: three-variable VAR
`run_route1(market_id, route1_panel, meta)` → dict of results.

In [5]:
def run_route1(market_id, route1_panel, meta):
    """
    B1.1 AIC lag selection
    B1.2 Granger causality (bidirectional)
    B1.3 FDR-corrected lag scan
    B1.4 Bootstrap IRF with CIs
    """
    mid     = market_id.lower()
    out_m   = OUTPUTS_DIR / mid
    out_r1  = out_m / 'route1'
    out_r1.mkdir(parents=True, exist_ok=True)

    tci_col = meta['tci_col']
    y       = route1_panel.copy()
    T       = len(y)
    max_lags = min(12, max(1, T // 10))
    log.info('[%s] Route1: T=%d months, max_lags=%d', mid.upper(), T, max_lags)

    # ── B1.1: AIC lag selection ───────────────────────────────────────────────
    var_model  = VAR(y)
    lag_result = var_model.select_order(maxlags=max_lags)
    p_opt      = lag_result.aic
    if p_opt == 0:
        p_opt = 1          # enforce minimum 1 lag
    log.info('[%s] Route1 VAR lag order (AIC): p=%d', mid.upper(), p_opt)

    var_fit = VAR(y).fit(maxlags=p_opt, ic=None)

    # ── B1.2: Granger causality (bidirectional) ───────────────────────────────
    def granger_pval(result, caused, causing):
        """Extract p-value from statsmodels VAR Granger test."""
        test = result.test_causality(caused, causing, kind='f')
        return test.pvalue

    gc_results = {}
    pairs = [
        ('ΔEPU → TCI', 'ΔEPU', tci_col),
        ('ΔTPU → TCI', 'ΔTPU', tci_col),
        ('ΔEPU+ΔTPU → TCI (joint)', ['ΔEPU', 'ΔTPU'], tci_col),
        ('TCI → ΔEPU', tci_col, 'ΔEPU'),
        ('TCI → ΔTPU', tci_col, 'ΔTPU'),
    ]
    for label, causing, caused in pairs:
        try:
            pv = granger_pval(var_fit, caused, causing)
            gc_results[label] = round(pv, 6)
            log.info('[%s] Granger %s p=%.4f', mid.upper(), label, pv)
        except Exception as e:
            gc_results[label] = np.nan
            _log_issue(mid, 'GRANGER', f'{label}: {e}')

    gc_df = pd.DataFrame.from_dict(gc_results, orient='index', columns=['p_value'])
    gc_df['significant_0.05'] = gc_df['p_value'] < 0.05
    gc_df.to_csv(out_r1 / 'granger_causality.csv')

    # ── B1.3: FDR-corrected lag scan ─────────────────────────────────────────
    scan_records = []
    for p_scan in range(1, max_lags + 1):
        try:
            vf_scan = VAR(y).fit(maxlags=p_scan, ic=None)
            rec = {'lag': p_scan}
            for epu_tpu in ['ΔEPU', 'ΔTPU']:
                pv = granger_pval(vf_scan, tci_col, epu_tpu)
                rec[f'{epu_tpu}→TCI_p'] = pv
            scan_records.append(rec)
        except Exception as e:
            _log_issue(mid, 'FDR_SCAN', f'lag={p_scan}: {e}')

    scan_df = pd.DataFrame(scan_records).set_index('lag')

    for col in ['ΔEPU→TCI_p', 'ΔTPU→TCI_p']:
        if col in scan_df.columns:
            reject = bh_correct(scan_df[col].dropna().values)
            scan_df[col.replace('_p', '_BH_reject')] = False
            scan_df.loc[scan_df[col].notna(), col.replace('_p', '_BH_reject')] = reject

    n_survive_epu = int(scan_df.get('ΔEPU→TCI_BH_reject', pd.Series(dtype=bool)).sum())
    n_survive_tpu = int(scan_df.get('ΔTPU→TCI_BH_reject', pd.Series(dtype=bool)).sum())
    log.info('[%s] FDR-surviving lags: EPU=%d, TPU=%d', mid.upper(), n_survive_epu, n_survive_tpu)
    scan_df.to_csv(out_r1 / 'fdr_lag_scan.csv')

    # ── B1.4: Bootstrap IRF ───────────────────────────────────────────────────
    irf_obj   = var_fit.irf(periods=IRF_HORIZON)
    irf_point = irf_obj.irfs               # (IRF_HORIZON+1, k, k) — point estimates
    col_names = list(y.columns)
    epu_idx   = col_names.index('ΔEPU')
    tpu_idx   = col_names.index('ΔTPU')
    tci_idx   = col_names.index(tci_col)

    # Bootstrap CIs via residual resampling
    rng       = np.random.default_rng(42)
    resids    = np.asarray(var_fit.resid)
    T_r, k    = resids.shape
    coefs     = np.asarray(var_fit.params)   # (k*p + 1, k) intercept + lags
    boot_irfs_epu = np.zeros((IRF_REPS, IRF_HORIZON + 1))
    boot_irfs_tpu = np.zeros((IRF_REPS, IRF_HORIZON + 1))

    for b in range(IRF_REPS):
        idx      = rng.integers(0, T_r, size=T_r)
        resid_b  = resids[idx]
        try:
            vf_b     = VAR(y + pd.DataFrame(
                np.vstack([np.zeros((p_opt, k)), resid_b[:T - p_opt]]),
                index=y.index, columns=y.columns
            )).fit(maxlags=p_opt, ic=None)
            irfs_b   = np.asarray(vf_b.irf(periods=IRF_HORIZON).irfs)
            boot_irfs_epu[b] = irfs_b[:, tci_idx, epu_idx]
            boot_irfs_tpu[b] = irfs_b[:, tci_idx, tpu_idx]
        except Exception:
            boot_irfs_epu[b] = np.nan
            boot_irfs_tpu[b] = np.nan

    irf_epu_point = np.asarray(irf_point)[:, tci_idx, epu_idx]
    irf_tpu_point = np.asarray(irf_point)[:, tci_idx, tpu_idx]
    ci_lo_epu = np.nanpercentile(boot_irfs_epu, 2.5,  axis=0)
    ci_hi_epu = np.nanpercentile(boot_irfs_epu, 97.5, axis=0)
    ci_lo_tpu = np.nanpercentile(boot_irfs_tpu, 2.5,  axis=0)
    ci_hi_tpu = np.nanpercentile(boot_irfs_tpu, 97.5, axis=0)

    horizons  = np.arange(IRF_HORIZON + 1)
    irf_df    = pd.DataFrame({
        'horizon':       horizons,
        'irf_epu':       irf_epu_point,
        'ci_lo_epu':     ci_lo_epu,
        'ci_hi_epu':     ci_hi_epu,
        'irf_tpu':       irf_tpu_point,
        'ci_lo_tpu':     ci_lo_tpu,
        'ci_hi_tpu':     ci_hi_tpu,
    }).set_index('horizon')
    irf_df.to_csv(out_r1 / 'irf.csv')

    log.info('[%s] Route1 complete. Outputs → %s', mid.upper(), out_r1)
    return {
        'lag_order':       p_opt,
        'var_fit':         var_fit,
        'granger':         gc_df,
        'fdr_scan':        scan_df,
        'n_survive_epu':   n_survive_epu,
        'n_survive_tpu':   n_survive_tpu,
        'irf':             irf_df,
        'tci_col':         tci_col,
        'col_names':       col_names,
    }

## Stage B — Route 2: thirteen-node VAR (11 sectors + ΔEPU + ΔTPU)

In [6]:
def run_route2(market_id, route2_panel, meta):
    """
    B2.1 AIC lag selection (typically p=1 given DOF constraints)
    B2.2 Generalised FEVD (Pesaran-Shin, order-invariant)
    B2.3 TO / FROM / NET directional measures
    B2.4 Policy-to-sector pairwise sensitivity table
    """
    mid    = market_id.lower()
    out_m  = OUTPUTS_DIR / mid
    out_r2 = out_m / 'route2'
    out_r2.mkdir(parents=True, exist_ok=True)

    z      = route2_panel.copy()
    T, N   = z.shape
    nodes  = list(z.columns)
    log.info('[%s] Route2: T=%d months, N=%d nodes', mid.upper(), T, N)

    # ── B2.1: AIC lag selection ───────────────────────────────────────────────
    max_lags_r2 = min(4, max(1, T // (N + 1)))
    lag_result  = VAR(z).select_order(maxlags=max_lags_r2)
    p_opt_r2    = lag_result.aic
    if p_opt_r2 == 0:
        p_opt_r2 = 1
    log.info('[%s] Route2 VAR lag order (AIC): p=%d (expected ~1 for monthly sample)',
             mid.upper(), p_opt_r2)

    var_fit_r2 = VAR(z).fit(maxlags=p_opt_r2, ic=None)

    # ── B2.2: Generalised FEVD ────────────────────────────────────────────────
    theta_raw  = compute_gfevd(var_fit_r2, HORIZON)
    theta_star = row_norm(theta_raw)

    # ── B2.3: Directional measures ────────────────────────────────────────────
    measures, tsi_r2 = directional_measures(theta_star, nodes)
    log.info('[%s] Route2 TCI (13-node): %.2f%%', mid.upper(), tsi_r2)
    log.info('[%s] NET(ΔEPU)=%.4f  NET(ΔTPU)=%.4f',
             mid.upper(),
             measures.loc['ΔEPU', 'NET'] if 'ΔEPU' in measures.index else np.nan,
             measures.loc['ΔTPU', 'NET'] if 'ΔTPU' in measures.index else np.nan)

    measures.to_csv(out_r2 / 'directional_measures_13node.csv')

    # Full 13×13 FEVD matrix
    cm_df = pd.DataFrame(theta_star, index=nodes, columns=nodes)
    cm_df.to_csv(out_r2 / 'fevd_matrix_13node.csv')

    # ── B2.4: Policy-to-sector pairwise table ─────────────────────────────────
    # Rows = policy nodes, cols = 11 GICS sectors
    sector_cols = [n for n in nodes if n not in ('ΔEPU', 'ΔTPU')]
    policy_rows = [n for n in ['ΔEPU', 'ΔTPU'] if n in nodes]

    policy_to_sector = pd.DataFrame(
        {sec: {pol: theta_star[nodes.index(pol), nodes.index(sec)] * 100
               for pol in policy_rows}
         for sec in sector_cols}
    )
    policy_to_sector.index.name = 'policy_node'
    policy_to_sector = policy_to_sector.round(4)
    policy_to_sector.to_csv(out_r2 / 'policy_to_sector.csv')

    log.info('[%s] Route2 complete. Outputs → %s', mid.upper(), out_r2)
    return {
        'lag_order':        p_opt_r2,
        'var_fit':          var_fit_r2,
        'fevd_matrix':      cm_df,
        'measures':         measures,
        'tci_r2':           tsi_r2,
        'policy_to_sector': policy_to_sector,
        'nodes':            nodes,
        'sector_cols':      sector_cols,
        'policy_rows':      policy_rows,
    }

## Market pipeline wrapper & main execution block

In [7]:
def market_policy_pipeline(market_id, cfg):
    """
    Full pipeline for one market (A0→A3, Route1, Route2).

    Parameters
    ----------
    market_id : str
        One of MARKETS ('china', 'us').
    cfg : dict
        MARKET_CONFIGS[market_id] — passed through to prepare_market_data.

    Returns
    -------
    r1 : dict   Route 1 results + meta
    r2 : dict   Route 2 results + meta
    """
    r1_panel, r2_panel, meta = prepare_market_data(market_id, cfg)
    r1 = run_route1(market_id, r1_panel, meta)
    r2 = run_route2(market_id, r2_panel, meta)
    r1['meta'] = meta
    r2['meta'] = meta
    return r1, r2


# ── Stages A & B: isolated per-market loop ────────────────────────────────────
# Each iteration is fully self-contained; no state leaks between markets.
market_results = {}   # container for cross-market retrieval in Stage C

for mkt in MARKETS:
    cfg = MARKET_CONFIGS[mkt]
    log.info('═' * 60)
    log.info('Processing %s …', mkt.upper())
    r1, r2 = market_policy_pipeline(mkt, cfg)
    market_results[mkt] = {'r1': r1, 'r2': r2}
    log.info('%s complete.', mkt.upper())

log.info('═' * 60)
log.info('All markets processed. Proceeding to Stage C.')

# ── Convenience aliases for Stage C readability (no recomputation) ─────────────
china_r1 = market_results['china']['r1']
china_r2 = market_results['china']['r2']
us_r1    = market_results['us']['r1']
us_r2    = market_results['us']['r2']

2026-06-26 14:21:23  INFO      ════════════════════════════════════════════════════════════
2026-06-26 14:21:23  INFO      Processing CHINA …
2026-06-26 14:21:23  INFO      [CHINA] ── Stage A: loading data ──


FileNotFoundError: Missing policy file: /Users/nath/Library/Mobile Documents/com~apple~CloudDocs/Desktop/CSAI_Y02_RESEARCH_INTERNSHIP/Modelling-Industry-Spillover-Effects-Using-Graph-Based-Methods/data/china/China_EPU.csv

## Stage C — Cross-market comparison (interpretation only)
Results for China and US are merged here **for display only**.  
No model is re-estimated and no pipeline variable is mutated.

In [ ]:
# ── C1: Side-by-side Granger causality table ─────────────────────────────────
china_gc = china_r1['granger'].rename(columns={'p_value': 'China_p', 'significant_0.05': 'China_sig'})
us_gc    = us_r1['granger'].rename(columns={'p_value': 'US_p', 'significant_0.05': 'US_sig'})
granger_compare = pd.concat([china_gc, us_gc], axis=1)

print('\n══ Granger Causality Comparison ══')
print(f'China Route1 VAR lag: p={china_r1["lag_order"]}  |  US Route1 VAR lag: p={us_r1["lag_order"]}')
display(granger_compare.round(4))
print(f'\nFDR-surviving lag tests (EPU→TCI): China={china_r1["n_survive_epu"]}  US={us_r1["n_survive_epu"]}')
print(f'FDR-surviving lag tests (TPU→TCI): China={china_r1["n_survive_tpu"]}  US={us_r1["n_survive_tpu"]}')

In [ ]:
# ── C2: IRF overlay — China vs US response of TCI to policy shock ─────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=False)
titles    = ['TCI response to 1σ ΔEPU shock', 'TCI response to 1σ ΔTPU shock']
irf_keys  = [('irf_epu', 'ci_lo_epu', 'ci_hi_epu'), ('irf_tpu', 'ci_lo_tpu', 'ci_hi_tpu')]
colors    = {'China': '#c0392b', 'US': '#2980b9'}

for ax, title, (pt_col, lo_col, hi_col) in zip(axes, titles, irf_keys):
    for market_label, r1 in [('China', china_r1), ('US', us_r1)]:
        idf = r1['irf']
        h   = idf.index
        c   = colors[market_label]
        ax.plot(h, idf[pt_col], color=c, lw=1.8, label=market_label)
        ax.fill_between(h, idf[lo_col], idf[hi_col], color=c, alpha=0.15)
    ax.axhline(0, color='k', lw=0.6, ls='--')
    ax.set_title(title, fontsize=10)
    ax.set_xlabel('Months ahead')
    ax.set_ylabel('Response of TCI')
    ax.legend(fontsize=9)
    ax.grid(axis='y', lw=0.4, alpha=0.5)

plt.suptitle('Impulse Response Functions: TCI to Policy Shocks', fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'irf_overlay.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/irf_overlay.png')

In [ ]:
# ── C3: Policy node NET transmitter bar chart ─────────────────────────────────
net_data = {}
for market_label, r2 in [('China', china_r2), ('US', us_r2)]:
    meas = r2['measures']
    for pol in ['ΔEPU', 'ΔTPU']:
        if pol in meas.index:
            net_data[(market_label, pol)] = meas.loc[pol, 'NET']
        else:
            net_data[(market_label, pol)] = np.nan

net_df = pd.Series(net_data).unstack(level=1)   # index=market, cols=policy

fig, ax = plt.subplots(figsize=(6, 4))
x       = np.arange(len(net_df.columns))
width   = 0.32
for i, (mkt, row) in enumerate(net_df.iterrows()):
    bars = ax.bar(x + i * width - width / 2, row.values, width, label=mkt,
                  color=['#c0392b', '#2980b9'][i], alpha=0.85)

ax.axhline(0, color='k', lw=0.7)
ax.set_xticks(x)
ax.set_xticklabels(net_df.columns, fontsize=10)
ax.set_ylabel('NET (%) — positive = net transmitter')
ax.set_title('Policy Node NET Directional Role (13-node VAR)', fontsize=10)
ax.legend()
ax.grid(axis='y', lw=0.4, alpha=0.5)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'net_policy_nodes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/net_policy_nodes.png')
print('\nNET table:')
display(net_df.round(4))

In [ ]:
# ── C4: Policy-to-sector sensitivity heatmap (China vs US) ───────────────────
china_pts = china_r2['policy_to_sector']
us_pts    = us_r2['policy_to_sector']

# Align columns to GICS order where available
sec_order = [s for s in GICS_SECTORS if s in china_pts.columns]

fig, axes = plt.subplots(1, 2, figsize=(15, 3.5))
for ax, pts, title in zip(axes,
                           [china_pts[sec_order], us_pts[sec_order]],
                           ['China: policy → sector spillover (%)', 'US: policy → sector spillover (%)']):
    sns.heatmap(
        pts,
        ax=ax, annot=True, fmt='.2f', cmap='YlOrRd',
        linewidths=0.4, linecolor='#cccccc',
        cbar_kws={'shrink': 0.75},
        xticklabels=[s[:9] for s in sec_order],
    )
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('')
    ax.tick_params(axis='x', labelsize=7, rotation=35)
    ax.tick_params(axis='y', labelsize=8)

plt.suptitle('Pairwise policy-node → sector FEVD contribution (θ̃, row-normalised, %)',
             fontsize=9, y=1.02)
plt.tight_layout()
plt.savefig(OUTPUTS_DIR / 'policy_sector_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: outputs/policy_sector_heatmap.png')

In [ ]:
# ── C5: Full 13-node directional table (both markets) ────────────────────────
print('\n══ China — 13-node directional measures (sorted by NET) ══')
print(f'Total Connectedness Index (Route2): {china_r2["tci_r2"]:.2f}%  |  VAR lag: p={china_r2["lag_order"]}')
display(china_r2['measures'].sort_values('NET', ascending=False).round(4))

print('\n══ US — 13-node directional measures (sorted by NET) ══')
print(f'Total Connectedness Index (Route2): {us_r2["tci_r2"]:.2f}%  |  VAR lag: p={us_r2["lag_order"]}')
display(us_r2['measures'].sort_values('NET', ascending=False).round(4))

In [ ]:
# ── C6: Stationarity summary (Stage A2) for both markets ─────────────────────
for mkt in MARKETS:
    r1   = market_results[mkt]['r1']
    meta = r1['meta']
    adf  = meta['adf']
    df_adf = pd.DataFrame(adf).T.rename(columns={'stationary': 'Stationary', 'p': 'ADF p-value'})
    market_label = mkt.capitalize()
    print(f'\n══ {market_label} — ADF results ══')
    print(f'Route1 window: {meta["r1_window"][0].date()} → {meta["r1_window"][1].date()}  N={meta["r1_N"]} months')
    print(f'Route2 window: {meta["r2_window"][0].date()} → {meta["r2_window"][1].date()}  N={meta["r2_N"]} months')
    display(df_adf.round(6))